In [1]:
import pandas as pd
from collections import defaultdict
from datetime import datetime

ODONTOGENIC_PATH = "/content/drive/MyDrive/odontogenic_codbook_EN.xlsx"
NON_ODONTOGENIC_PATH = "/content/drive/MyDrive/non_odontogenic_codbook_EN.xlsx"
NEUROLOGICAL_REDFLAGS_PATH = "/content/drive/MyDrive/neurological_redflags_codbook_EN.xlsx"


def load_codbook_from_excel(path: str, category: str) -> pd.DataFrame:
    """
    Load a codbook (odontogenic / non-odontogenic / neurological red flags)
    and normalize it to a common schema.
    """
    df = pd.read_excel(path)

    required_cols = {"diagnostic", "label_en", "description_en"}
    assert required_cols.issubset(df.columns), f"Missing columns in {path}"

    cols_to_keep = [c for c in ["diagnostic", "code", "label_en", "description_en", "pattern"] if c in df.columns]
    df = df[cols_to_keep].copy()

    df.rename(
        columns={
            "diagnostic": "disease",
            "label_en": "symptom_label",
            "description_en": "symptom_description",
        },
        inplace=True,
    )

    for col in ["disease", "symptom_label", "symptom_description"]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip()

    if "pattern" in df.columns:
        df["pattern"] = df["pattern"].astype(str).str.strip()

    df["category"] = category

    return df


def generate_df_from_dict(d: dict) -> pd.DataFrame:
    """
    Utility kept from the old version (might be useful for synthetic cases).
    """
    rows = []
    for disease, symptoms in d.items():
        for s in symptoms:
            rows.append({"disease": disease, "symptom_label": s})
    return pd.DataFrame(rows)


df_odontogenic = load_codbook_from_excel(ODONTOGENIC_PATH, "odontogenic")
df_non_odontogenic = load_codbook_from_excel(NON_ODONTOGENIC_PATH, "non_odontogenic")
df_neurological = load_codbook_from_excel(NEUROLOGICAL_REDFLAGS_PATH, "neurological_red_flag")

df = pd.concat([df_odontogenic, df_non_odontogenic, df_neurological], ignore_index=True)

df.drop_duplicates(inplace=True)

print(f"Total number of rows: {len(df)}")
print(f"Number of unique diseases: {df['disease'].nunique()}")
print(
    "Average number of symptoms per disease: "
    f"{df.groupby('disease')['symptom_label'].nunique().mean():.2f}"
)

print("\nNumber of diseases / category:")
print(df.groupby("category")["disease"].nunique())

print("\nMost frequent symptoms (by label):")
print(df["symptom_label"].value_counts().head(6))

print("\nRandom examples (category, disease, symptom_label):")
print(df.sample(10, random_state=0)[["category", "disease", "symptom_label"]])


Total number of rows: 132
Number of unique diseases: 23
Average number of symptoms per disease: 5.74

Number of diseases / category:
category
neurological_red_flag     1
non_odontogenic          11
odontogenic              11
Name: disease, dtype: int64

Most frequent symptoms (by label):
symptom_label
Pain on tooth percussion    7
Throbbing tooth pain        6
Spontaneous pain            5
Swelling                    4
Pain on percussion          3
Fever                       3
Name: count, dtype: int64

Random examples (category, disease, symptom_label):
            category                            disease  \
93   non_odontogenic                          Sinusitis   
66   non_odontogenic  Implant – inferior alveolar nerve   
26       odontogenic       Chronic apical periodontitis   
8        odontogenic  Symptomatic irreversible pulpitis   
30       odontogenic               Acute apical abscess   
91   non_odontogenic                          Sinusitis   
109  non_odontogenic    

In [2]:
from collections import defaultdict

def to_mapping(df):
    """
    Build a dictionary:
    disease → list of symptom dictionaries
    where each symptom has: label, description, pattern, category.
    """
    mapping = defaultdict(list)

    for _, row in df.iterrows():
        symptom = {
            "label": row["symptom_label"],
            "description": row.get("symptom_description", ""),
            "pattern": row.get("pattern", None),
            "category": row["category"],
        }
        mapping[row["disease"]].append(symptom)

    return dict(mapping)


mapping = to_mapping(df)

print(f"\nNumber of unique diseases in mapping: {len(mapping)}\n")

for i, (disease, symptoms) in enumerate(mapping.items()):
    if i >= 3:
        break
    print(f"Disease: {disease}")
    print(f"Number of symptoms: {len(symptoms)}")
    print("Symptoms:")
    for s in symptoms:
        print(f"  - {s['label']}")
    print("\n" + "-"*50 + "\n")



Number of unique diseases in mapping: 23

Disease: Reversible pulpitis
Number of symptoms: 5
Symptoms:
  - Short cold pain
  - No spontaneous pain
  - No nocturnal pain
  - No pain on percussion
  - Lingering pain to cold

--------------------------------------------------

Disease: Symptomatic irreversible pulpitis
Number of symptoms: 5
Symptoms:
  - Spontaneous pain
  - Lingering pain to cold
  - Nocturnal pain
  - Heat-induced pain
  - Mild percussion pain

--------------------------------------------------

Disease: Asymptomatic irreversible pulpitis
Number of symptoms: 5
Symptoms:
  - No pain
  - Normal percussion
  - Deep caries present
  - Thermal irritation minimal
  - Spontaneous pain

--------------------------------------------------



In [3]:
from dataclasses import dataclass, field
import random
from typing import List, Dict

@dataclass
class Case:
    disease_truth: str
    symptoms_truth: List[dict]

def init_case(mapping: Dict[str, List[dict]]) -> Case:
    """
    Select a random disease and return it with all associated symptoms.
    Compatible with the new dataset structure.
    """
    chosen_disease = random.choice(list(mapping.keys()))
    chosen_symptoms = mapping[chosen_disease]   # list of dicts
    return Case(
        disease_truth=chosen_disease,
        symptoms_truth=chosen_symptoms
    )

# Test
case = init_case(mapping)
print("Disease:", case.disease_truth)
print("Symptoms:")

for s in case.symptoms_truth:
    print(f"  - {s['label']}")


Disease: Denture pain
Symptoms:
  - Pain only when denture is worn
  - Pain disappears without denture
  - Mucosal ulcerations under denture
  - Pain on tooth percussion
  - Throbbing tooth pain
  - Nocturnal pain
  - Localized mucosal pain


In [4]:
!pip -q install groq
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

resp = client.chat.completions.create(
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    messages=[{"role": "user", "content": "Tell me a short joke about dentists."}],
)

print(resp.choices[0].message.content)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 3.5 MB/s eta 0:00:00
Here's one:

Why did the dentist become a baker?

Because he kneaded the dough! (get it?)


In [5]:
from dataclasses import dataclass, field
import json, re
import os
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])


@dataclass
class State:
    history: List[str] = field(default_factory=list)
    slots: dict = field(default_factory=dict)

    def get_context_for_prompt(self, limit: int = 8) -> str:
        if not self.history:
            return "The conversation has just started."

        # prima replica
        first_turn = self.history[0]

        # ultimele replici
        recent_turns = self.history[-limit:]

        if len(self.history) <= limit:
            return "\n".join(self.history)

        separator = "\n... [middle of conversation omitted] ...\n"
        # daca prima replica e deja in limit, nu o dublam
        if first_turn in recent_turns:
             return "\n".join(recent_turns)

        return f"{first_turn}{separator}" + "\n".join(recent_turns)


def build_patient_system_prompt(case: Case, max_new_clues: int = 1) -> str:
    """
    Build the system prompt for the virtual patient.
    Uses the ground-truth disease and its true symptoms (from case.symptoms_truth).
    """
    symptom_labels = sorted({s["label"] for s in case.symptoms_truth})
    symptoms_text = ", ".join(symptom_labels)

    return f"""
You are a VIRTUAL HUMAN PATIENT talking to a dental medicine student.
Your goal is to help the student practice identifying symptoms, NOT to tell them the diagnosis.

INTERNAL INFO (only for you, never reveal directly):
- True underlying disease (hidden diagnosis): {case.disease_truth}
- Full list of your true symptoms (internal, do not list them all at once): {symptoms_text}

This internal symptom list is your ONLY source of truth.
You MUST NEVER claim (in your words) to have any pain, sensitivity,
discomfort, weird feeling or problem that is NOT in this list.

IMPORTANT:
The student will often use natural, informal language
(e.g. "does it hurt when you drink something cold?", "does it hurt when you chew on that side?",
"is your jaw clicking?", "do you feel pressure under your eyes?").
You should usually understand what symptom they refer to.

====================
BEHAVIOR RULES
====================

1. Never mention the diagnosis or disease names (caries, pulpitis, gingivitis, stroke, etc.)
   and do NOT give medical explanations, causes, treatments, or advice.

2. Speak like a normal person, using informal, simple language
   ("it hurts", "I noticed", "it feels sharp", "it's annoying", "I'm not sure").
   However:
   - You MUST NOT describe any area, action, or situation as painful, sensitive,
     uncomfortable, weird, or bothering you UNLESS it corresponds to a real symptom
     in your internal list.
   - If the student asks about something that is NOT one of your real symptoms
     (for example cold sensitivity when you have no cold/heat-related symptoms),
     you MUST clearly say that it does NOT bother you or that you have NOT noticed
     any problem with that.

3. In your FIRST message of the conversation, you MUST:
   - Start with a short, natural greeting (e.g. “Hi”, “Hello”, “Hi doctor”).
   - Then briefly and vaguely explain what brought you here today.
   - Do NOT mention many specific symptoms yet.
   - Everything you say must still be consistent with your true symptom list.


4. When the student asks about a SPECIFIC SYMPTOM, you MUST:
   - Identify which real symptom the question corresponds to.
   - If the symptom *is in your true internal list*:
         → answer naturally and set slot_updates[symptom_label] = true.
   - If the symptom is *NOT in your true internal list*:
         → answer clearly that this does NOT cause you problems and set
           slot_updates[symptom_label] = false.
   - Your natural-language answer MUST be consistent with slot_updates:
         if you set a symptom to false, your text must NOT describe that symptom
         as painful, sensitive, uncomfortable, or bothering you.
   - You MUST NOT leave slot_updates empty when the question clearly refers to a symptom.
   - DO NOT say “I'm not sure what you mean” for obvious symptom questions involving:
       pain, hurting, sensitivity, cold, hot, chewing, biting, pressure, swelling,
       taste changes, smell, bleeding, clicking, popping, numbness, electric shock pain,
       headache, facial pressure, jaw opening, jaw closing, etc.
     Only ask for clarification when the question is TRULY ambiguous.

5. If the question is very general ("Do you have any other problems?"),
   you may reveal at most {max_new_clues} NEW real symptom(s).

6. Remain consistent: never contradict your previous answers.

7. Do NOT repeat symptoms you've already mentioned,
   and NEVER invent symptoms or discomforts that are NOT in your real symptom list.

8. Each answer must be short and natural (1–3 sentences).

9. slot_updates RULE:
   - Use true if the asked-about symptom is genuinely present in your internal list.
   - Use false if the student clearly asks about a symptom you do NOT have.
   - If the question is vague, you MAY leave slot_updates empty.
   - The KEYS in slot_updates MUST be the exact English symptom labels
     from your internal list when possible (e.g. "Cold sensitivity", "Pain on chewing",
     "Food impaction pain", "Bad taste / halitosis").
   - If the student describes a symptom using different words (e.g. “cold drink pain”),
     map it to the closest real symptom label.

10. Avoid using the exact phrase “I'm not sure what you mean” repeatedly.
    If clarification is needed, vary your wording politely.

11. If the student talks about unrelated topics (life, jokes, exams, etc.),
    politely redirect back to your symptom discussion.

====================
OUTPUT FORMAT
====================

You MUST return ONLY a valid JSON object, EXACTLY in this structure:

{{
  "assistant_text": "the patient's natural-language answer",
  "slot_updates": {{
    "symptom_name_1": true/false,
    "symptom_name_2": true/false
  }}
}}

Do NOT add any text outside the JSON. No explanations, no comments, no extra keys.
""".strip()




def build_patient_context(state: State, case: Case) -> str:
    conversation_summary = state.get_context_for_prompt(limit=8)

    confirmed = [k for k, v in state.slots.items() if v is True]
    denied = [k for k, v in state.slots.items() if v is False]

    all_true_symptoms = [s["label"] for s in case.symptoms_truth]
    unrevealed_true = [s for s in all_true_symptoms if s not in confirmed]

    context = f"""
CURRENT CONVERSATION CONTEXT:
Summary of dialogue:
{conversation_summary}

Symptoms CONFIRMED so far: {', '.join(confirmed) or 'none'}
Symptoms DENIED so far: {', '.join(denied) or 'none'}
True symptoms not yet explicitly mentioned (do NOT reveal them directly to the student):
{', '.join(unrevealed_true) or 'none'}

Instructions:
- Keep your answers coherent with this history.
- Do NOT contradict symptoms you already confirmed or denied.
- Try to be cooperative: if the question clearly refers to one of your symptoms
  (even if phrased informally), answer it instead of saying you don't understand.
- Only ask for clarification when the question is genuinely ambiguous.
""".strip()

    return context


def build_user_prompt(user_msg: str) -> str:
    """
    Wrap the student's message into a clear instruction for the model.
    """
    return f"""
The student asks you: {user_msg}

Follow all the rules from the system and context.

You MUST return ONLY a JSON object with EXACTLY this structure:
{{
  "assistant_text": "your short, realistic answer as the patient",
  "slot_updates": {{
    "some_symptom_key": true/false
  }}
}}
- Use 'slot_updates' to mark specific symptoms that the student is really asking about.
- When they ask about pain or sensitivity with HOT or COLD food or drinks,
  you should normally treat that as a clear question about thermal sensitivity/pain.
- When they ask about chewing, biting, jaw movement, or clicking,
  treat that as a question about function-related symptoms.

Do NOT add any text outside this JSON. No explanations, no comments, no extra keys.
""".strip()


def safe_json_parse(content: str):
    """
    Try to extract and parse a JSON object from the model's response.
    Applies some light cleanup if needed.
    """
    content = content.strip()

    match = re.search(r"\{[\s\S]*\}", content)
    if not match:
        raise RuntimeError(f"Could not find any JSON object in the response:\n{content}")
    json_part = match.group(0)

    json_part = json_part.replace("True", "true").replace("False", "false")
    json_part = json_part.replace("’", "'").replace("„", '"').replace("”", '"')
    json_part = re.sub(r",\s*}", "}", json_part)
    json_part = re.sub(r",\s*]", "]", json_part)

    try:
        return json.loads(json_part)
    except Exception as e:
        raise RuntimeError(
            f"Failed to parse JSON even after cleanup:\n{json_part}\nError: {e}"
        )


def llm_call_groq(
    system_prompt: str,
    context_prompt: str,
    user_prompt: str,
    model: str = "meta-llama/llama-4-scout-17b-16e-instruct",
) -> dict:
    """
    Single call to the Groq LLM.
    We send SYSTEM + CONTEXT + USER blocks all in one 'user' message,
    because the API here uses only 'messages' with roles.
    """
    prompt = (
        f"[SYSTEM]\n{system_prompt}\n\n"
        f"[CONTEXT]\n{context_prompt}\n\n"
        f"[USER]\n{user_prompt}\n"
    )
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=300,
        )
        content = resp.choices[0].message.content.strip()
    except Exception as e:
        raise RuntimeError(f"Groq API error: {e}")

    return safe_json_parse(content)


In [6]:
def step(
    case: Case,
    state: State,
    user_msg: str,
    max_new_clues: int = 1,
    model: str = "meta-llama/llama-4-scout-17b-16e-instruct",
) -> dict:

    system_prompt = build_patient_system_prompt(case, max_new_clues=max_new_clues)
    context_prompt = build_patient_context(state, case)
    user_prompt = build_user_prompt(user_msg)

    out = llm_call_groq(system_prompt, context_prompt, user_prompt, model=model)

    text = out.get("assistant_text", "").strip()
    slot_updates = out.get("slot_updates", {}) or {}

    true_symptom_norms = {
        s["label"].lower().replace(" ", "_") for s in case.symptoms_truth
    }
    filtered_updates = {}
    for k, v in slot_updates.items():
        k_norm = k.lower().replace(" ", "_")
        if v is False and k_norm in true_symptom_norms:
            continue
        filtered_updates[k] = v
    slot_updates = filtered_updates

    addition = f"Student: {user_msg} | Patient: {text}"
    state.history.append(addition)

    if isinstance(slot_updates, dict):
        state.slots.update(slot_updates)

    text_lower = text.lower()
    true_labels = [s["label"] for s in case.symptoms_truth]
    label_norm_map = {lbl.lower().replace(" ", "_"): lbl for lbl in true_labels}
    revealed = set()
    for lbl in true_labels:
        if lbl.lower() in text_lower:
            revealed.add(lbl)
    for k, v in state.slots.items():
        if not v: continue
        k_norm = k.lower().replace(" ", "_")
        if k_norm in label_norm_map:
            revealed.add(label_norm_map[k_norm])

    return {
        "assistant_text": text,
        "slot_updates": slot_updates,
        "summary": state.get_context_for_prompt(),
        "revealed_symptoms": list(revealed),
    }

In [46]:
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"
MAX_NEW_CLUES = 1

def start_new_case():
    """Start a new case with a random diagnosis and show only the patient's first message."""
    global case, state
    case = init_case(mapping)
    state = State()
    print("New case started.")
    print("Type /help for the list of commands.")

    first_question = "Hi, what brings you here today?"

    out = step(
        case,
        state,
        first_question,
        max_new_clues=MAX_NEW_CLUES,
        model=MODEL,
    )


    print("Patient:", out["assistant_text"])

    return case, state



HELP_TEXT = """Available commands:
/help       - show this help
/truth      - show the hidden diagnosis and its true symptoms (debug)
/summary    - show the conversation summary
/slots      - show current slots (confirmed/denied symptoms)
/revealed   - show true symptoms that are considered revealed
/new        - start a new case
/diagnose X - submit final diagnosis (ends session + feedback)
/exit       - end the chat session
"""


def show_truth():
    symptom_labels = sorted({s["label"] for s in case.symptoms_truth})
    print("HIDDEN DIAGNOSIS:", case.disease_truth)
    print("True symptoms:", ", ".join(symptom_labels))


def show_summary():
    """
    Shows exactly what text is sent to the LLM as context.
    It calls get_context_for_prompt() to simulate the sliding window.
    """
    if not state.history:
        print("SUMMARY: (Conversation hasn't started yet)")
        return

    llm_view = state.get_context_for_prompt(limit=8)

    print("\n" + "="*50)
    print(" LLM CONTEXT VIEW (What the model sees) ")
    print("="*50)
    print(llm_view)
    print("="*50)
    print(f"Stats: Total turns in history: {len(state.history)} | Window limit: 8")
    print("="*50 + "\n")


def show_slots():
    print("SLOTS:", state.slots or "(none)")


def show_revealed():
    """
    A symptom is considered 'revealed' if:
    - it appears in the patient's text at some point (approximated here via slots=True)
      OR
    - it was explicitly marked as true in slots.
    We recompute this from the current state rather than storing it separately.
    """
    true_labels = [s["label"] for s in case.symptoms_truth]
    label_norm_map = {
        lbl.lower().replace(" ", "_"): lbl
        for lbl in true_labels
    }

    revealed = set()
    for k, v in state.slots.items():
        if not v:
            continue
        k_norm = k.lower().replace(" ", "_")
        if k_norm in label_norm_map:
            revealed.add(label_norm_map[k_norm])

    print("REVEALED:", sorted(revealed) or "(none)")


def evaluate_final_diagnosis(case: Case, user_guess: str, model: str):
    """
    Evaluates the final diagnosis using the LLM as a Professor.
    Strictly follows the provided symptom list as the absolute ground truth.
    """

    print(f"\nEvaluating final diagnosis: '{user_guess}'...\n")

    system_prompt = """
You are an expert Dental Professor grading a student's FINAL DIAGNOSIS.

CRITICAL RULE: The "True Hidden Diagnosis" and "True Symptoms List" provided to you are the ABSOLUTE TRUTH for this specific case.
Even if your external medical knowledge suggests that a certain symptom is "atypical" or "rare" for that disease, YOU MUST ACCEPT IT as a fact for this patient.

INSTRUCTIONS:
1. Accept synonyms (e.g., "Caries" == "Tooth Decay").
2. Address the student DIRECTLY with "You".
3. Determine if the diagnosis is "CORRECT", "PARTIALLY CORRECT", or "WRONG".
4. FEEDBACK STYLE:
   - Be supportive and brief.
   - Do NOT lecture the student on general theory.
   - Do NOT contradict the provided symptom list. Do NOT say things like "However, this symptom is usually not present" or "Note that this is atypical".
   - Simply explain that the diagnosis is correct because it matches the patient's reported symptoms (the ones provided in the list).

OUTPUT FORMAT (JSON ONLY):
{
  "status": "CORRECT" | "PARTIALLY_CORRECT" | "WRONG",
  "feedback": "Your direct feedback..."
}
"""

    user_prompt = f"""
True Hidden Diagnosis: "{case.disease_truth}"
True Symptoms List: "{', '.join([s['label'] for s in case.symptoms_truth])}"
Student's Guess: "{user_guess}"

Evaluate based ONLY on the provided True Symptoms List. Return the JSON.
"""

    try:
        out = llm_call_groq(system_prompt, "", user_prompt, model=model)

        status = out.get("status", "WRONG").upper()
        feedback = out.get("feedback", "No feedback provided.")

        print("="*60)
        print(f"🎓 EVALUATION RESULT: {status}")
        print("="*60)
        print(f"Professor Feedback: {feedback}")
        print("-" * 60)

        if status != "CORRECT":
            print(f"\nThe correct diagnosis was: {case.disease_truth}")
            symptom_list = ", ".join([s['label'] for s in case.symptoms_truth])
            print(f"True symptoms were: {symptom_list}")
        else:
            print("\nCONGRATULATIONS! You identified the correct diagnosis!")

    except Exception as e:
        print(f"Evaluation error: {e}")
        print(f"The correct diagnosis was: {case.disease_truth}")


def chat_loop():
    while True:
        try:
            user_msg = input("\nYou: ").strip()
        except EOFError:
            break

        if not user_msg:
            continue

        cmd = user_msg.lower()
        if cmd == "/help":
            print(HELP_TEXT)
            continue
        if cmd == "/exit":
            print("Session closed.")
            break
        if cmd == "/truth":
            show_truth()
            continue
        if cmd == "/summary":
            show_summary()
            continue
        if cmd == "/slots":
            show_slots()
            continue
        if cmd == "/revealed":
            show_revealed()
            continue
        if cmd == "/new":
            start_new_case()
            continue
        if cmd.startswith("/diagnose"):
            guess = user_msg[len("/diagnose"):].strip()

            if not guess:
                print("Please type the diagnosis name. Example: /diagnose Disease")
                continue

            evaluate_final_diagnosis(case, guess, model=MODEL)

            print("\nSession ended.")
            break

        try:
            out = step(
                case,
                state,
                user_msg,
                max_new_clues=MAX_NEW_CLUES,
                model=MODEL,
            )
            print("Patient:", out["assistant_text"])
        except Exception as e:
            print("Error in step():", e)


start_new_case()
chat_loop()


New case started.
Type /help for the list of commands.
Patient: Hi, I've been having some issues with my tooth lately and I thought it would be good to get it checked out.

You: /truth
HIDDEN DIAGNOSIS: Symptomatic irreversible pulpitis
True symptoms: Heat-induced pain, Lingering pain to cold, Mild percussion pain, Nocturnal pain, Spontaneous pain

You: does it bother you when you drink something cold?
Patient: Yeah, drinking something cold can be a bit uncomfortable for me.

You: /slots
SLOTS: {'Lingering pain to cold': True}

You: what about drinking something hot like a coffee?
Patient: Drinking something hot can be a bit uncomfortable for me too.

You: /slots
SLOTS: {'Lingering pain to cold': True, 'Heat-induced pain': True}

You: do you have some pain when i tap your tooth?
Patient: When you tap on it, it does hurt a bit.

You: /slots
SLOTS: {'Lingering pain to cold': True, 'Heat-induced pain': True, 'Mild percussion pain': True}

You: is your pain continuos?
Patient: No, it's not